# Phase 4: Bilateral Market-Making vs Baseline Comparison

This notebook trains a bilateral market-making RL agent and compares it against a simple baseline.

**Workflow**:
1. Setup environment and dependencies
2. Clone/pull latest code from repository
3. Implement SymmetricFixedSpread baseline agent
4. Train bilateral RL agent (200 iterations, quick config)
5. Evaluate bilateral agent (1000 episodes)
6. Evaluate baseline agent (1000 episodes)
7. Compare metrics and visualize results

**Runtime**: ~30-40 minutes (GPU)

---

## Step 0: Clear cache and setup repository


In [14]:
!git clone https://github.com/SalmanSattar24/rtle_parallelized.git /content/rtle_parallelized
%cd /content/rtle_parallelized

fatal: destination path '/content/rtle_parallelized' already exists and is not an empty directory.
/content/rtle_parallelized


## Step 1: Setup Environment

In [15]:
# Install dependencies (Colab)
import sys
import subprocess

# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("[INFO] Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("[INFO] Running locally")

# Install dependencies if in Colab
if IN_COLAB:
    print("\n[INSTALL] Installing PyTorch and dependencies...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "gymnasium", "tensorboard", "tyro"])
    print("[OK] Dependencies installed")
else:
    print("[CHECK] Verifying dependencies...")
    try:
        import torch
        import gymnasium
        import tensorboard
        print("[OK] All dependencies present")
    except ImportError as e:
        print(f"[WARNING] Missing dependency: {e}")

[INFO] Running in Google Colab

[INSTALL] Installing PyTorch and dependencies...
[OK] Dependencies installed


## Step 2: Clone/Pull Repository

In [16]:
import os
import subprocess
import shutil

# Setup directory paths
if IN_COLAB:
    # Clone to /content in Colab
    repo_dir = "/content/rtle_parallelized"
    if os.path.exists(repo_dir):
        print(f"[PULL] Updating existing repo at {repo_dir}")
        os.chdir(repo_dir)
        subprocess.run(["git", "pull"], check=True, capture_output=True)
    else:
        print(f"[CLONE] Cloning repository to {repo_dir}")
        subprocess.run(
            ["git", "clone", "https://github.com/SalmanSattar24/rtle_parallelized.git", repo_dir],
            check=True,
            capture_output=True
        )
else:
    # Local path
    repo_dir = "C:/All-Code/CSCI-566/rtle_parallelized"
    print(f"[INFO] Using local repository at {repo_dir}")
    if os.path.exists(os.path.join(repo_dir, ".git")):
        os.chdir(repo_dir)
        print("[PULL] Updating repository...")
        subprocess.run(["git", "pull"], capture_output=True)

os.chdir(repo_dir)
print(f"\n[OK] Working directory: {os.getcwd()}")
print(f"[VERIFY] Repository structure:")
for folder in ["simulation", "rl_files", "tests", "limit_order_book"]:
    path = os.path.join(repo_dir, folder)
    print(f"  {'[OK]' if os.path.exists(path) else '[MISS]'} {folder}/")

[PULL] Updating existing repo at /content/rtle_parallelized

[OK] Working directory: /content/rtle_parallelized
[VERIFY] Repository structure:
  [OK] simulation/
  [OK] rl_files/
  [OK] tests/
  [OK] limit_order_book/


## Step 3.5: Update Repository to Latest Version

In [17]:
print("=" * 70)
print("STEP 3.5: Updating Repository to Latest Version")
print("=" * 70)

print("\nFetching latest commits from GitHub...")
import subprocess
subprocess.run(["git", "pull", "origin", "master"], cwd=repo_dir, check=True)

print("\nâœ“ Repository updated!")
print("=" * 70)


STEP 3.5: Updating Repository to Latest Version

Fetching latest commits from GitHub...

âœ“ Repository updated!


In [18]:
# Sync guard: verify Colab/runtime commit before expensive runs
import subprocess
from pathlib import Path

repo_path = Path(repo_dir)
try:
    current_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=repo_path).decode().strip()
    current_branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=repo_path).decode().strip()
    print(f"[GIT] Branch: {current_branch}")
    print(f"[GIT] Commit: {current_commit}")
except Exception as e:
    current_commit = None
    print(f"[WARN] Could not read git commit: {e}")

# Optional strict pin: set this to the commit you expect after push
EXPECTED_COMMIT = None  # e.g., "a338e21"

if EXPECTED_COMMIT is not None and current_commit is not None:
    assert current_commit == EXPECTED_COMMIT, (
        f"Commit mismatch! expected={EXPECTED_COMMIT}, got={current_commit}. "
        "Run repo update cell before training/eval."
    )
    print("[OK] Commit hash matches expected version")
else:
    print("[INFO] EXPECTED_COMMIT is not set; skipping strict commit assertion")

[GIT] Branch: master
[GIT] Commit: 5b4db8c
[INFO] EXPECTED_COMMIT is not set; skipping strict commit assertion


In [19]:
import numpy as np

print("=" * 70)
print("PHASE 6: AVELLANEDA-STOIKOV (AS) BASELINE CALCULATION")
print("=" * 70)

def compute_as_quotes(mid_price, inventory, time_left, sigma=2.0, gamma=0.1, k=1.5):
    # reservation_price = mid_price - q * gamma * sigma^2 * (T - t)
    # spread = (2 / gamma) * ln(1 + gamma / k) + gamma * sigma^2 * (T - t)
    
    res_price = mid_price - inventory * gamma * (sigma**2) * time_left
    spread = (2 / gamma) * np.log(1 + gamma / k) + gamma * (sigma**2) * time_left
    
    bid_price = np.floor(res_price - spread / 2)
    ask_price = np.ceil(res_price + spread / 2)
    
    return bid_price, ask_price

# Example AS quotes
mid = 1000
inv = 5
t_left = 0.5
b, a = compute_as_quotes(mid, inv, t_left)
print(f"[AS TEST] Mid: {mid} | Inv: {inv} | Bid: {b} | Ask: {a} | Spread: {a-b}")


PHASE 6: AVELLANEDA-STOIKOV (AS) BASELINE CALCULATION
[AS TEST] Mid: 1000 | Inv: 5 | Bid: 998.0 | Ask: 1000.0 | Spread: 2.0


## Step 4: Import Libraries and Setup Paths

In [20]:
import sys
import os
import numpy as np
import torch
import torch.nn as nn
import time
from typing import Optional, Dict, List, Tuple
import matplotlib.pyplot as plt

# Add paths for imports
sys.path.insert(0, repo_dir)
sys.path.insert(0, os.path.join(repo_dir, "simulation"))
sys.path.insert(0, os.path.join(repo_dir, "rl_files"))
sys.path.insert(0, os.path.join(repo_dir, "limit_order_book"))

# Import core modules
from simulation.market_gym import Market
from rl_files.actor_critic import BilateralAgentLogisticNormal

print("[OK] All imports successful")
print(f"[INFO] Using device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
print("[OK] Random seeds set")

[OK] All imports successful
[INFO] Using device: cuda
[OK] Random seeds set


In [21]:
print("="*70)
print("FORCE FRESH REPO LOAD (clear cached imports)")
print("="*70)

# Force reimport of all modules - clear cached modules
import sys
to_remove = [key for key in sys.modules if 'simulation' in key or 'rl_files' in key or 'limit_order_book' in key]
for key in to_remove:
    del sys.modules[key]

print("[OK] Cleared cached modules")
print("="*70 + "\n")

FORCE FRESH REPO LOAD (clear cached imports)
[OK] Cleared cached modules



## Step 4: Implement SymmetricFixedSpread Baseline Agent

In [22]:
from typing import Tuple

class SymmetricFixedSpreadAgent:
    """
    Baseline market-making agent:
    - Posts 1 lot at best bid (passive buy)
    - Posts 1 lot at best ask (passive sell)
    - Returns TUPLE format to match environment expectations
    - Simple, static strategy
    """

    def __init__(self, action_space_dim: int = 7):
        self.action_space_dim = action_space_dim
        # Action allocation:
        # [market%, L1%, L2%, L3%, L4%, L5%, inactive%]
        # For baseline: 0% market, 100% L1 (1 lot at best)
        self.action = np.zeros(action_space_dim)
        self.action[1] = 1.0  # Place 100% at level 1 (best bid/ask)

    def get_action(self, obs: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Return fixed action tuple (bid_action, ask_action) regardless of observation.
        
        Both bid and ask use the same allocation (symmetric fixed spread).
        Environment expects tuple of two 7-dim arrays.
        """
        return self.action.copy(), self.action.copy()

print("[OK] SymmetricFixedSpreadAgent defined (returns tuple format)")

[OK] SymmetricFixedSpreadAgent defined (returns tuple format)


## Step 5: Configuration

In [23]:
# Paper-Faithful Configuration (Cheridito & Weiss 2026)
# Start with NOISE environment to validate pipeline, then move to flow/strategic

TRAIN_CONFIG = {
    'market_env': 'noise',       # Start simple: noise-only market
    'execution_agent': 'rl_agent',
    'volume': 10,                # Paper uses 10 lots with I_max=10
    'seed': 42,
    'terminal_time': 150,        # Paper: 150 seconds
    'time_delta': 15,            # Paper: 15s intervals → 10 time steps
    'drop_feature': 'drift',     # Paper default
    'inventory_max': 10,         # Paper: tight cap experiment
    'penalty_weight': 0.0,       # NO quadratic penalty (paper doesn't use one)
}

EVAL_CONFIG = {
    'market_env': 'noise',
    'execution_agent': 'rl_agent',
    'volume': 10,
    'seed': 100,
    'terminal_time': 150,
    'time_delta': 15,
    'drop_feature': 'drift',
    'inventory_max': 10,
    'penalty_weight': 0.0,
}

EVAL_EPISODES = 1000

print("[OK] Paper-Faithful Config loaded (noise, I_max=10, volume=10, T=150, dt=15)")


[OK] Paper-Faithful Config loaded (noise, I_max=10, volume=10, T=150, dt=15)


## Step 6: Create Market Environment and Agents

In [24]:
print("[SETUP] Creating market environment...")
market_env = Market(TRAIN_CONFIG)
obs, info = market_env.reset(seed=42)

print(f"[INFO] Observation shape: {obs.shape}")
print(f"[INFO] Action space: {market_env.action_space.shape}")

# Create a simple wrapper so BilateralAgentLogisticNormal can work with Market
class EnvWrapper:
    def __init__(self, env):
        self.env = env
        # Add single_observation_space and single_action_space for agent init
        self.single_observation_space = env.observation_space
        self.single_action_space = env.action_space
    
    # Proxy methods to underlying environment
    def reset(self, seed=None):
        return self.env.reset(seed=seed)
    
    def step(self, action):
        return self.env.step(action)

market = EnvWrapper(market_env)

print("\n[SETUP] Creating bilateral RL agent...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bilateral_agent = BilateralAgentLogisticNormal(market).to(device)
print(f"[OK] Bilateral agent on {device}")

print("\n[SETUP] Creating baseline agent...")
baseline_agent = SymmetricFixedSpreadAgent(market_env.action_space.shape[0])
print(f"[OK] Baseline agent ready")

print("\n[SETUP] All agents created successfully")

[SETUP] Creating market environment...
[INFO] Observation shape: (44,)
[INFO] Action space: (7,)

[SETUP] Creating bilateral RL agent...
[OK] Bilateral agent on cuda

[SETUP] Creating baseline agent...
[OK] Baseline agent ready

[SETUP] All agents created successfully


## Step 7: Train Bilateral Agent


In [25]:
import torch

def project_action_quota(a, current_inv, side="bid", inventory_max=10, q_max_base=10):
    """
    Paper-faithful quota projection.
    Q_max = min(q_max_base, I_max - |I(t)|) per side per step.
    
    This mechanically prevents inventory overflow through the action space,
    no artificial penalties needed.
    """
    x = torch.clamp(a, min=1e-8)
    
    # Paper: Q_max = min(q_max_base, I_max - |I(t)|)
    if side == "bid":
        room = max(0.0, inventory_max - current_inv)  # Room to buy more
    else:  # ask
        room = max(0.0, inventory_max + current_inv)  # Room to sell more
    
    q_max = min(q_max_base, room)
    
    if q_max <= 0:
        # No room: force everything to hold (last bucket)
        result = torch.zeros_like(x)
        result[..., -1] = 1.0
        return result
    
    # Scale active mass (non-hold buckets) proportional to quota
    volume = 10  # Must match TRAIN_CONFIG['volume']
    active_mass_limit = q_max / volume
    
    active_mass = torch.sum(x[..., :-1], dim=-1, keepdim=True)
    scale = torch.minimum(torch.ones_like(active_mass), active_mass_limit / (active_mass + 1e-8))
    
    x[..., :-1] *= scale
    # Renormalize to ensure valid simplex
    total = torch.sum(x, dim=-1, keepdim=True)
    x = x / (total + 1e-8)
    
    return x

print("[OK] Paper-faithful quota projection loaded (Q_max = min(10, I_max - |I(t)|))")


[OK] Paper-faithful quota projection loaded (Q_max = min(10, I_max - |I(t)|))


In [26]:
print("=" * 70)
print("STEP 7: TRAIN BILATERAL AGENT (Paper-Faithful PPO)")
print("=" * 70)
print()

import copy
import time
import numpy as np
import torch
import torch.nn.functional as F

# === Paper Hyperparameters (Cheridito & Weiss 2026) ===
NUM_TRAIN_ITERS = 400       # Paper: 400 gradient steps
EPISODES_PER_ITER = 64      # Paper: 1280 trajectories; we use 64 as compute-feasible approximation
PPO_EPOCHS = 4
MINIBATCH_SIZE = 256

BASE_LR = 5e-4              # Paper: 0.0005
CLIP_EPS = 0.20
VF_COEF = 0.5               # Paper standard
MAX_GRAD_NORM = 0.5         # Paper: clip gradients at 0.5

GAMMA_TRAIN = 1.0           # Paper: undiscounted (gamma=1)
GAE_LAMBDA = 1.0            # Paper: full returns (lambda=1)

# Paper: variance schedule sigma_init=1.0 -> sigma_final=0.1
ENT_COEF_START = 0.25
ENT_COEF_END = 0.05

# Fresh start
bilateral_agent = BilateralAgentLogisticNormal(market).to(device)
optimizer = torch.optim.Adam(bilateral_agent.parameters(), lr=BASE_LR, eps=1e-5)
training_returns = []
training_losses = []
start_time = time.time()

best_val_score = -float('inf')
best_val_return = -float('inf')
best_state_dict = copy.deepcopy(bilateral_agent.state_dict())

def quick_eval(agent, episodes=120, seed_base=20000):
    vals = []
    for i in range(episodes):
        cfg = dict(EVAL_CONFIG)
        cfg['seed'] = seed_base + i
        m_raw = Market(cfg)
        market_wrap = EnvWrapper(m_raw) # Correct wrapper use
        obs, _ = market_wrap.reset()
        ep_ret = 0.0
        current_inventory = 0

        while True:
            obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            with torch.no_grad():
                # deterministic_action returns (bid, ask) tuple directly
                bid_action, ask_action = agent.deterministic_action(obs_tensor)
                bid_action = project_action_quota(bid_action, current_inventory, side="bid", inventory_max=EVAL_CONFIG['inventory_max'])
                ask_action = project_action_quota(ask_action, current_inventory, side="ask", inventory_max=EVAL_CONFIG['inventory_max'])
                env_action = (bid_action[0].cpu().numpy(), ask_action[0].cpu().numpy())
            obs, reward, terminated, truncated, info = market_wrap.step(env_action)
            ep_ret += float(reward)
            current_inventory = info.get("net_inventory", 0)
            if terminated or truncated:
                break
        vals.append(ep_ret)
    vals = np.array(vals)
    cvar5 = np.mean(np.sort(vals)[:max(1, len(vals)//20)])
    outlier = np.mean(vals < -200.0)
    return float(np.mean(vals)), float(np.std(vals)), float(cvar5), float(outlier)

for iteration in range(NUM_TRAIN_ITERS):
    iter_start = time.time()

    # Entropy/variance schedule (linear decay)
    frac = iteration / max(NUM_TRAIN_ITERS - 1, 1)
    ent_coef = ENT_COEF_START + (ENT_COEF_END - ENT_COEF_START) * frac

    # Variance schedule (paper: sigma from 1.0 to 0.1)
    if hasattr(bilateral_agent, 'set_variance'):
        new_var = 1.0 - 0.9 * frac  # 1.0 -> 0.1
        bilateral_agent.set_variance(new_var)

    obs_buf, bid_buf, ask_buf, old_logprob_buf, adv_buf, ret_buf = [], [], [], [], [], []

    for episode in range(EPISODES_PER_ITER):
        cfg = dict(TRAIN_CONFIG)
        cfg['seed'] = iteration * 10000 + episode
        m_raw = Market(cfg)
        market_wrap = EnvWrapper(m_raw)
        obs, _ = market_wrap.reset()
        current_inventory = 0

        ep_obs, ep_bid, ep_ask, ep_logprobs, ep_values, ep_rewards = [], [], [], [], [], []
        ep_return = 0.0

        while True:
            obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            # FIXED UNPACKING: get_action_and_value returns ((bid, ask), log_prob, entropy, value)
            (bid_action, ask_action), log_prob, entropy, value = bilateral_agent.get_action_and_value(obs_tensor)

            # Paper-faithful quota projection
            bid_action = project_action_quota(bid_action, current_inventory, side="bid", inventory_max=TRAIN_CONFIG['inventory_max'])
            ask_action = project_action_quota(ask_action, current_inventory, side="ask", inventory_max=TRAIN_CONFIG['inventory_max'])

            # Recompute log prob after projection
            # FIXED UNPACKING here as well
            _, log_prob_proj, _, _ = bilateral_agent.get_action_and_value(obs_tensor, action=(bid_action, ask_action))

            env_action = (bid_action[0].detach().cpu().numpy(), ask_action[0].detach().cpu().numpy())
            obs_next, reward, terminated, truncated, info = market_wrap.step(env_action)

            # Update inventory
            current_inventory = info.get("net_inventory", 0)

            # Paper-faithful reward
            reward = float(reward)

            ep_obs.append(obs_tensor.squeeze(0).detach())
            ep_bid.append(bid_action.squeeze(0).detach())
            ep_ask.append(ask_action.squeeze(0).detach())
            ep_logprobs.append(log_prob_proj.squeeze().detach())
            ep_values.append(value.squeeze().detach())
            ep_rewards.append(reward)

            ep_return += reward
            obs = obs_next

            if terminated or truncated:
                break

        rewards_t = torch.tensor(ep_rewards, dtype=torch.float32, device=device)
        values_t = torch.stack(ep_values)
        adv_t = torch.zeros_like(rewards_t)
        gae = 0.0
        next_value = 0.0

        for t in reversed(range(len(ep_rewards))):
            delta = rewards_t[t] + GAMMA_TRAIN * next_value - values_t[t]
            gae = delta + GAMMA_TRAIN * GAE_LAMBDA * gae
            adv_t[t] = gae
            next_value = values_t[t]

        ret_t = adv_t + values_t

        obs_buf.extend(ep_obs)
        bid_buf.extend(ep_bid)
        ask_buf.extend(ep_ask)
        old_logprob_buf.extend(ep_logprobs)
        adv_buf.extend([a.detach() for a in adv_t])
        ret_buf.extend([r.detach() for r in ret_t])

        training_returns.append(ep_return)

    b_obs = torch.stack(obs_buf)
    b_bid = torch.stack(bid_buf)
    b_ask = torch.stack(ask_buf)
    b_old_logprobs = torch.stack(old_logprob_buf)
    b_advantages = torch.stack(adv_buf)
    b_returns = torch.stack(ret_buf)

    # Standard advantage normalisation
    adv_mean = b_advantages.mean()
    adv_std = b_advantages.std()
    if torch.isnan(adv_std) or adv_std < 1e-6:
        b_advantages = b_advantages - adv_mean
    else:
        b_advantages = (b_advantages - adv_mean) / (adv_std + 1e-8)

    batch_size = b_obs.shape[0]
    mb_size = min(MINIBATCH_SIZE, batch_size)
    inds = np.arange(batch_size)

    iter_losses = []

    for _ in range(PPO_EPOCHS):
        np.random.shuffle(inds)
        for start in range(0, batch_size, mb_size):
            mb_inds = inds[start:start + mb_size]
            mb_inds_t = torch.tensor(mb_inds, dtype=torch.long, device=device)

            mb_obs = b_obs[mb_inds_t]
            mb_actions = (b_bid[mb_inds_t], b_ask[mb_inds_t])
            mb_old_logprobs = b_old_logprobs[mb_inds_t]
            mb_adv = b_advantages[mb_inds_t]
            mb_returns = b_returns[mb_inds_t]

            _, new_logprob, entropy, new_value = bilateral_agent.get_action_and_value(mb_obs, action=mb_actions)
            new_value = new_value.squeeze()

            # Clean PPO loss
            log_ratio = new_logprob - mb_old_logprobs
            ratio = torch.exp(log_ratio)

            surr1 = ratio * mb_adv
            surr2 = torch.clamp(ratio, 1.0 - CLIP_EPS, 1.0 + CLIP_EPS) * mb_adv
            actor_loss = -torch.min(surr1, surr2).mean()

            value_loss = F.smooth_l1_loss(new_value, mb_returns)
            entropy_bonus = entropy.mean()

            total_loss = actor_loss + VF_COEF * value_loss - ent_coef * entropy_bonus

            optimizer.zero_grad(set_to_none=True)
            total_loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(bilateral_agent.parameters(), max_norm=MAX_GRAD_NORM)

            if torch.isnan(total_loss) or torch.isinf(total_loss) or torch.isnan(grad_norm):
                continue

            optimizer.step()
            iter_losses.append(float(total_loss.item()))

    mean_loss = float(np.mean(iter_losses)) if len(iter_losses) > 0 else float('nan')
    training_losses.append(mean_loss)

    if (iteration + 1) % 20 == 0:
        val_mean, val_std, val_cvar5, val_outlier = quick_eval(bilateral_agent, episodes=120, seed_base=30000 + iteration * 100)
        val_score = val_mean - 0.20 * val_std - 100.0 * val_outlier
        if val_score > best_val_score:
            best_val_score = val_score
            best_val_return = val_mean
            best_state_dict = copy.deepcopy(bilateral_agent.state_dict())
            tag = " [BEST]"
        else:
            tag = ""
    else:
        val_mean, val_std, val_cvar5, val_outlier, val_score = float('nan'), float('nan'), float('nan'), float('nan'), float('nan')
        tag = ""

    if (iteration + 1) % 20 == 0 or iteration < 5:
        elapsed_iter = time.time() - iter_start
        elapsed_total = time.time() - start_time
        avg_return_20 = np.mean(training_returns[-20:]) if len(training_returns) >= 20 else np.mean(training_returns)
        current_var = bilateral_agent.variance if hasattr(bilateral_agent, 'variance') and bilateral_agent.variance is not None else float('nan')
        print(
            f"[{iteration+1:3d}/{NUM_TRAIN_ITERS}] {elapsed_iter:5.1f}s | "
            f"Total: {elapsed_total:6.1f}s | Avg20: {avg_return_20:8.2f} | "
            f"Loss: {mean_loss:8.4f} | Ent: {ent_coef:6.4f} | Var: {current_var:5.3f} | "
            f"Val120: {val_mean:8.2f}\u00b1{val_std:6.2f} | CVaR5: {val_cvar5:8.2f} | Out<-200: {val_outlier:6.3f} | "
            f"Score: {val_score:8.2f}{tag}"
        )

bilateral_agent.load_state_dict(best_state_dict)

print(f"\n[OK] Training complete in {time.time() - start_time:.1f}s")
print(f"[INFO] Training return (last 20): {np.mean(training_returns[-20:]):.4f}")
print(f"[INFO] Training loss (last 20): {np.mean(training_losses[-20:]):.4f}")
print(f"[INFO] Best validation mean return (120 eps): {best_val_return:.4f}")
print(f"[INFO] Best validation score: {best_val_score:.4f}")
print("=" * 70 + "\n")


STEP 7: TRAIN BILATERAL AGENT (Paper-Faithful PPO)

[  1/400]  25.6s | Total:   25.6s | Avg20:    -0.97 | Loss:  -3.8364 | Ent: 0.2500 | Var: 1.000 | Val120:      nan±   nan | CVaR5:      nan | Out<-200:    nan | Score:      nan
[  2/400]  25.3s | Total:   50.9s | Avg20:    -0.62 | Loss:  -3.8888 | Ent: 0.2495 | Var: 1.000 | Val120:      nan±   nan | CVaR5:      nan | Out<-200:    nan | Score:      nan
[  3/400]  25.0s | Total:   76.0s | Avg20:    -0.78 | Loss:  -3.8936 | Ent: 0.2490 | Var: 1.000 | Val120:      nan±   nan | CVaR5:      nan | Out<-200:    nan | Score:      nan
[  4/400]  24.8s | Total:  100.8s | Avg20:    -1.26 | Loss:  -3.8512 | Ent: 0.2485 | Var: 1.000 | Val120:      nan±   nan | CVaR5:      nan | Out<-200:    nan | Score:      nan
[  5/400]  25.0s | Total:  125.8s | Avg20:    -0.35 | Loss:  -3.7649 | Ent: 0.2480 | Var: 1.000 | Val120:      nan±   nan | CVaR5:      nan | Out<-200:    nan | Score:      nan
[ 20/400]  67.5s | Total:  544.0s | Avg20:    -0.90 | Loss:  -3

## Step 8: Evaluate Bilateral Agent (Final Best Model)
# This performs a large-scale evaluation of the trained agent to compare against baseline.


In [27]:
import torch
import numpy as np

print("=" * 70)
print("STEP 8: EVALUATE BILATERAL AGENT (RL)")
print("=" * 70)

# Ensure best weights are loaded (redundant but safe)
if 'best_state_dict' in locals():
    bilateral_agent.load_state_dict(best_state_dict)

EVAL_EPISODES = 1000
bilateral_returns = []
bilateral_inventories = []

for i in range(EVAL_EPISODES):
    cfg = dict(EVAL_CONFIG)
    cfg['seed'] = 50000 + i 
    m_raw = Market(cfg)
    market_wrap = EnvWrapper(m_raw)
    obs, _ = market_wrap.reset()
    ep_ret = 0.0
    current_inventory = 0

    while True:
        obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            # Evaluation uses DETERMINISTIC actions
            bid_action, ask_action = bilateral_agent.deterministic_action(obs_tensor)
            
            # Quota projection is strictly applied for consistency
            bid_action = project_action_quota(bid_action, current_inventory, side="bid", inventory_max=EVAL_CONFIG['inventory_max'])
            ask_action = project_action_quota(ask_action, current_inventory, side="ask", inventory_max=EVAL_CONFIG['inventory_max'])
            
            env_action = (bid_action[0].cpu().numpy(), ask_action[0].cpu().numpy())
            
        obs, reward, terminated, truncated, info = market_wrap.step(env_action)
        ep_ret += float(reward)
        current_inventory = info.get("net_inventory", 0)
        
        if terminated or truncated:
            break
            
    bilateral_returns.append(ep_ret)
    bilateral_inventories.append(abs(current_inventory))
    
    if (i + 1) % 200 == 0:
        print(f"[{i+1:4d}/{EVAL_EPISODES}] Mean return: {np.mean(bilateral_returns):.4f} | Abs Inv: {np.mean(bilateral_inventories):.4f}")

bilateral_returns = np.array(bilateral_returns)
bilateral_inventories = np.array(bilateral_inventories)

print(f"\n[OK] RL Evaluation complete: {np.mean(bilateral_returns):.4f} +/- {np.std(bilateral_returns):.4f}")


## Step 9: Evaluate Baseline Agent


In [28]:
print("=" * 70)
print(f"STEP 9: EVALUATE BASELINE (Fixed Spread)")
print("=" * 70)

baseline_agent = SymmetricFixedSpreadAgent(spread=1)
baseline_returns = []
baseline_inventories = []

for i in range(EVAL_EPISODES):
    cfg = dict(EVAL_CONFIG)
    cfg['seed'] = 50000 + i
    m_raw = Market(cfg)
    market_wrap = EnvWrapper(m_raw)
    obs, _ = market_wrap.reset()
    ep_ret = 0.0
    current_inventory = 0
    
    while True:
        action = baseline_agent.get_action(obs)
        obs, reward, terminated, truncated, info = market_wrap.step(action)
        ep_ret += float(reward)
        current_inventory = info.get("net_inventory", 0)
        if terminated or truncated:
            break
            
    baseline_returns.append(ep_ret)
    baseline_inventories.append(abs(current_inventory))

baseline_returns = np.array(baseline_returns)
baseline_inventories = np.array(baseline_inventories)

print(f"\n[OK] Baseline Evaluation complete: {np.mean(baseline_returns):.4f} +/- {np.std(baseline_returns):.4f}")


[EVAL] Evaluating baseline agent on 1000 episodes...

[ 100/1000] Episodes/sec: 3.0 | Mean return: -0.4930 | Std: 1.6233
[ 200/1000] Episodes/sec: 3.0 | Mean return: -0.5410 | Std: 1.5355
[ 300/1000] Episodes/sec: 3.0 | Mean return: -0.5977 | Std: 1.5401
[ 400/1000] Episodes/sec: 3.0 | Mean return: -0.5857 | Std: 1.5047
[ 500/1000] Episodes/sec: 3.0 | Mean return: -0.5656 | Std: 1.4975
[ 600/1000] Episodes/sec: 3.0 | Mean return: -0.5582 | Std: 1.4805
[ 700/1000] Episodes/sec: 3.0 | Mean return: -0.5396 | Std: 1.4786
[ 800/1000] Episodes/sec: 3.0 | Mean return: -0.5230 | Std: 1.4863
[ 900/1000] Episodes/sec: 3.0 | Mean return: -0.5354 | Std: 1.4909
[1000/1000] Episodes/sec: 3.0 | Mean return: -0.5240 | Std: 1.4922

[OK] Baseline evaluation complete in 332.5s
[INFO] Mean return: -0.5240 +/- 1.4922
[INFO] Mean terminal inventory: 10.0000


## Step 10: Compare Results


In [29]:
# Compute statistics for comparison
import pandas as pd

stats_data = {
    'Metric': ['Mean Return', 'Std Return', 'CVaR (5%)', 'Outliers (<-200)', 'Mean Abs Inv'],
    'Bilateral RL': [
        np.mean(bilateral_returns),
        np.std(bilateral_returns),
        np.mean(np.sort(bilateral_returns)[:max(1, len(bilateral_returns)//20)]),
        np.mean(bilateral_returns < -200.0),
        np.mean(bilateral_inventories)
    ],
    'Baseline': [
        np.mean(baseline_returns),
        np.std(baseline_returns),
        np.mean(np.sort(baseline_returns)[:max(1, len(baseline_returns)//20)]),
        np.mean(baseline_returns < -200.0),
        np.mean(baseline_inventories)
    ]
}

df_stats = pd.DataFrame(stats_data)
display(df_stats.style.background_gradient(subset=['Bilateral RL', 'Baseline'], cmap='RdYlGn'))


NameError: name 'bilateral_returns' is not defined

## Step 11: Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set premium aesthetic
plt.style.use('seaborn-v0_8-muted')
sns.set_context("talk")
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
colors = ['#4C72B0', '#C44E52', '#55A868', '#8172B3']

# 1. Return Distributions
sns.histplot(bilateral_returns, ax=axes[0, 0], color=colors[0], label='Bilateral RL', kde=True, element="step")
sns.histplot(baseline_returns, ax=axes[0, 0], color=colors[1], label='Baseline', kde=True, element="step")
axes[0, 0].set_title('Return Distribution Comparison')
axes[0, 0].set_xlabel('Profit / Loss')
axes[0, 0].legend()

# 2. Cumulative Returns (Box Plot)
axes[0, 1].boxplot([bilateral_returns, baseline_returns], labels=['Bilateral RL', 'Baseline'], patch_artist=True)
axes[0, 1].set_title('Return Range & Outliers')
axes[0, 1].set_ylabel('Profit / Loss')

# 3. Training Curve (if training_returns exists)
if 'training_returns' in locals() and len(training_returns) > 0:
    training_ma = pd.Series(training_returns).rolling(window=20).mean()
    axes[1, 0].plot(training_ma, color=colors[2], linewidth=3, label='Training Moving Avg (20)')
    axes[1, 0].set_title('Training Progress')
    axes[1, 0].set_xlabel('Episode Index')
    axes[1, 0].set_ylabel('Reward')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()
else:
    axes[1, 0].text(0.5, 0.5, "No training curve data", ha='center')

# 4. Inventory Management
axes[1, 1].hist(bilateral_inventories, bins=11, alpha=0.7, color=colors[0], label='RL Inventory', density=True)
axes[1, 1].hist(baseline_inventories, bins=11, alpha=0.7, color=colors[1], label='Baseline Inventory', density=True)
axes[1, 1].set_title('Terminal Inventory Density')
axes[1, 1].set_xlabel('Absolute Inventory Value')
axes[1, 1].legend()

plt.suptitle(f"Stabilized Bilateral MM Performance Analysis ({TRAIN_CONFIG['market_env'].upper()})", fontsize=24, y=1.02)
plt.tight_layout()
plt.savefig('bilateral_mm_success_report.png', dpi=150, bbox_inches='tight')
plt.show()

print("[OK] Performance report and plots generated.")


## Step 12: Summary and Next Steps

In [ ]:
print("\n" + "="*70)
print("PHASE 4 SIMPLIFIED: EXECUTION COMPLETE")
print("="*70)
print()
print("RESULTS SUMMARY")
print("-" * 70)

train_iters_used = NUM_TRAIN_ITERS if 'NUM_TRAIN_ITERS' in globals() else TRAIN_PARAMS['num_iterations']
print(f"Training iterations:        {train_iters_used}")
print(f"Evaluation episodes:        {EVAL_EPISODES}")
print()
print(f"Bilateral RL Agent:")
print(f"  Mean return:              {bilateral_stats['mean_return']:>10.4f}")
print(f"  Std deviation:            {bilateral_stats['std_return']:>10.4f}")
print(f"  Terminal inventory:       {bilateral_stats['mean_inventory']:>10.4f}")
print()
print(f"Baseline Agent (SymmetricFixedSpread):")
print(f"  Mean return:              {baseline_stats['mean_return']:>10.4f}")
print(f"  Std deviation:            {baseline_stats['std_return']:>10.4f}")
print(f"  Terminal inventory:       {baseline_stats['mean_inventory']:>10.4f}")
print()
print(f"Performance Gap:            {improvement:>10.4f} ({relative_improvement:+.1f}%)")
print("-" * 70)
print()
if improvement > 0:
    print("[SUCCESS] Bilateral RL agent demonstrates improvement over simple baseline!")
    print()
    print("Key findings:")
    print(f"  1. RL agent achieves {improvement:.4f} better PnL per episode")
    print(f"  2. {'Better' if bilateral_stats['std_return'] < baseline_stats['std_return'] else 'Similar'} variance control")
    print(f"  3. {'Improved' if bilateral_stats['mean_inventory'] < baseline_stats['mean_inventory'] else 'Similar'} inventory management")
else:
    print(f"[INFO] Baseline performs better. This may indicate:")
    print(f"  1. Need for more training iterations (more than {train_iters_used})")
    print(f"  2. Hyperparameter tuning required")
    print(f"  3. Different environment complexity needed")

print()
print("PHASE 4 EXPANDED (Optional):")
print("-" * 70)
print("To extend this analysis:")
print("  1. Add more baseline agents (TWAP, Avellaneda-Stoikov)")
print("  2. Train on larger batch (400+ iterations)")
print("  3. Compare across multiple environments")
print("  4. Analyze learned trading strategy")
print("  5. Extract quote depth vs time-to-expiry patterns")
print()
print("PHASE 5: Documentation and Results")
print("-" * 70)
print("  1. Generate comparison tables")
print("  2. Create detailed analysis report")
print("  3. Document implementation findings")
print("  4. Package results for publication")
print()
print("="*70)
print("\n[OK] Phase 4 Simplified Complete!")

## Optional: Save Results

In [ ]:
# Save results to file
import json

results = {
    'bilateral': bilateral_stats,
    'baseline': baseline_stats,
    'improvement': {
        'absolute': float(improvement),
        'percentage': float(relative_improvement),
    },
    'config': {
        'train_iterations': TRAIN_PARAMS['num_iterations'],
        'eval_episodes': EVAL_EPISODES,
        'env_type': TRAIN_CONFIG['market_env'],
        'inventory_max': TRAIN_CONFIG['inventory_max'],
    },
}

with open('phase4_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("[OK] Results saved to 'phase4_results.json'")
print()
print("[INFO] You can download these files:")
print("  - phase4_comparison.png (visualization)")
print("  - phase4_results.json (raw data)")

In [ ]:
# Diagnostics snapshot (auto-generated)
import numpy as np

def summarize(name, arr):
    a = np.asarray(arr, dtype=float)
    return {
        'n': int(len(a)),
        'mean': float(np.mean(a)),
        'std': float(np.std(a)),
        'p05': float(np.percentile(a, 5)),
        'p50': float(np.percentile(a, 50)),
        'p95': float(np.percentile(a, 95)),
        'min': float(np.min(a)),
        'max': float(np.max(a)),
    }

print('=== TRAINING ===')
if 'training_returns' in globals() and len(training_returns) > 0:
    tr = np.asarray(training_returns, dtype=float)
    print('training_returns:', summarize('training_returns', tr))
    print('train_tail_mean_20:', float(np.mean(tr[-20:])))
else:
    print('training_returns not available')

if 'training_losses' in globals() and len(training_losses) > 0:
    tl = np.asarray(training_losses, dtype=float)
    print('training_losses:', summarize('training_losses', tl))
    print('loss_tail_mean_20:', float(np.mean(tl[-20:])))
else:
    print('training_losses not available')

print('\n=== EVALUATION ===')
if 'bilateral_returns' in globals() and 'baseline_returns' in globals():
    br = np.asarray(bilateral_returns, dtype=float)
    ba = np.asarray(baseline_returns, dtype=float)
    print('bilateral:', summarize('bilateral', br))
    print('baseline :', summarize('baseline', ba))
    print('improvement_abs:', float(np.mean(br) - np.mean(ba)))
    denom = abs(np.mean(ba)) if np.mean(ba) != 0 else 1.0
    print('improvement_pct:', float(100.0 * (np.mean(br) - np.mean(ba)) / denom))
    print('bilateral_outlier_rate_r<-500:', float(np.mean(br < -500)))
    print('baseline_outlier_rate_r<-500 :', float(np.mean(ba < -500)))
else:
    print('evaluation arrays not available')

if 'bilateral_inventories' in globals() and 'baseline_inventories' in globals():
    bi = np.asarray(bilateral_inventories, dtype=float)
    bj = np.asarray(baseline_inventories, dtype=float)
    print('bilateral_inventory:', summarize('bilateral_inventory', bi))
    print('baseline_inventory :', summarize('baseline_inventory', bj))

In [ ]:

print("=" * 70)
print("PHASE 6 VERIFICATION: CIRCUIT BREAKER 'CRASH TEST'")
print("=" * 70)

# 1. Setup a Stress-Test Agent that only BUYS to force an inventory breach
class ForceBreachAgent:
    def __init__(self, action_dim=7):
        self.buy_action = np.zeros(action_dim)
        self.buy_action[0] = 1.0 # 100% Market Buy
        self.inactive = np.zeros(action_dim)
        self.inactive[-1] = 1.0

    def get_action(self, obs):
        # Always Market Buy, Inactive on Ask
        return self.buy_action.copy(), self.inactive.copy()

# 2. Run Episode in 'Strategic' environment
test_cfg = copy.deepcopy(TRAIN_CONFIG)
test_cfg['inventory_max'] = 30 # absolute limit
test_cfg['market_env'] = 'strategic'

env = Market(test_cfg)
agent = ForceBreachAgent()
obs, info = env.reset(seed=42)

total_reward = 0
steps = 0
triggered = False

while True:
    action = agent.get_action(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    
    if info.get('circuit_breaker', False):
        triggered = True
        print(f"[SUCCESS] Circuit Breaker Triggered at Step {steps}!")
        print(f"[INFO] Final Inventory: {info['net_inventory']}")
        print(f"[INFO] Reward for this step: {reward}")
        break
        
    if terminated or truncated:
        break

print(f"\n[RESULT] Triggered: {triggered} | Total Reward: {total_reward:.2f}")
if triggered and -51.0 <= reward <= -49.0:
    print("VERIFICATION PASSED: Simulator correctly caps tail-risk at -50.0")
else:
    print("VERIFICATION FAILED: Check market_gym.py implementation")
print("=" * 70)
